# Full-turn FBP: original detector stacks

Retain the full detector stacks from the original `test_reconstruct_FBP3d_tomobar_i12_dataset5`
and `test_reconstruct_FBP3d_tomobar_i13_dataset3` cases. The worked examples elsewhere
include a four-row I12 subset; this notebook keeps all fifteen I12 detector rows
and all three I13 rows, the original fixed overlaps, angle selections and output checks.

Data: [Zenodo record 17252190](https://zenodo.org/records/17252190). Set
`HTTOMOLIBGPU_DATA_DIR` to use an archive directory other than `Zenodo` in the checkout.
Install `.[validation]` and run serially with a CUDA GPU:

```bash
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/reconstruction/transferred/full_turn.ipynb
```

The source snapshot is recorded in the transfer overview in the Validation section.
Original checks cover shape, float32 dtype and C layout; image quality is visual review.
Allocation failures are recorded as failed cases and the next dataset still runs.
Inputs are not reduced to fit the GPU. Numeric failures are enforced after figures
are saved, and do not become accepted references. The documentation renders saved outputs.


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path
from math import isclose

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pytest
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print(
    json.dumps(
        {
            "python": platform.python_version(),
            "packages": {
                name: version(name)
                for name in (
                    "httomolibgpu",
                    "tomobar",
                    "astra-toolbox",
                    "numpy",
                    "matplotlib",
                )
            },
            "cupy": cp.__version__,
            "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
            "cuda_driver": cp.cuda.runtime.driverGetVersion(),
            "gpu": (
                props["name"].decode()
                if isinstance(props["name"], bytes)
                else props["name"]
            ),
        },
        indent=2,
    )
)
results = []


def force_clean_gpu_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def load_dataset(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    print(
        json.dumps(
            {
                "dataset": path.name,
                "zenodo_record": "17252190",
                "sha256": digest.hexdigest(),
            },
            indent=2,
        )
    )
    with np.load(path) as archive:
        if name == "i12_dataset3":
            return tuple(
                cp.asarray(archive[key]) for key in ("proj1", "proj2", "flats", "darks")
            )
        return (
            cp.asarray(archive["projdata"]),
            archive["angles"].copy(),
            cp.asarray(archive["flats"]),
            cp.asarray(archive["darks"]),
        )


def plot_comparison(before, after, title, label="Negative log transmission"):
    before, after = cp.asnumpy(before), cp.asnumpy(after)
    residual = before - after
    lo, hi = np.percentile(before[np.isfinite(before)], [1, 99])
    limit = float(np.percentile(np.abs(residual[np.isfinite(residual)]), 99)) or 1
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    for ax, image, name, cmap, vmin, vmax in (
        (axes[0], before, "Before", "gray", lo, hi),
        (axes[1], after, "After", "gray", lo, hi),
        (axes[2], residual, "Removed component", "coolwarm", -limit, limit),
    ):
        im = ax.imshow(image, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set(title=name, xlabel="Detector column", ylabel="Projection index")
        fig.colorbar(im, ax=ax, label=label)
    fig.suptitle(title)
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Transferred regressions failed: " + json.dumps(failed, indent=2)
    print(f"All {len(results)} transferred cases passed.")

In [ ]:
from httomolibgpu.recon.algorithm import (
    FBP3d_tomobar,
    LPRec3d_tomobar,
    FISTA3d_tomobar,
    ADMM3d_tomobar,
)
from httomolibgpu.misc.morph import sino_360_to_180

current_case = None


def record_check(expression, passed):
    results.append(
        {"test": current_case, "assertion": expression, "passed": bool(passed)}
    )


def check_allclose(*args, **kwargs):
    try:
        assert_allclose(*args, **kwargs)
    except AssertionError as exc:
        results.append(
            {
                "test": current_case,
                "assertion": "assert_allclose",
                "observed": float(args[0]),
                "expected": float(args[1]),
                "tolerances": kwargs,
                "passed": False,
                "details": str(exc),
            }
        )
    else:
        results.append(
            {
                "test": current_case,
                "assertion": "assert_allclose",
                "observed": float(args[0]),
                "expected": float(args[1]),
                "tolerances": kwargs,
                "passed": True,
            }
        )


def report_reconstruction(recon_data):
    print(
        json.dumps(
            {
                "test": current_case,
                "shape": list(recon_data.shape),
                "dtype": str(recon_data.dtype),
                "sum_float32": float(np.sum(recon_data)),
                "minimum": float(np.min(recon_data)),
                "maximum": float(np.max(recon_data)),
            },
            indent=2,
        )
    )
    image = recon_data[:, recon_data.shape[1] // 2, :]
    lo, hi = np.percentile(image, [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    width = image.shape[0]
    for ax, view, title in (
        (axes[0], image[::2, ::2], "Middle reconstructed slice"),
        (
            axes[1],
            image[width // 4 : 3 * width // 4, width // 4 : 3 * width // 4],
            "Central crop",
        ),
    ):
        im = ax.imshow(view, cmap="gray", vmin=lo, vmax=hi)
        ax.set(
            title=title,
            xlabel="Slice column (display pixel)",
            ylabel="Slice row (display pixel)",
        )
        fig.colorbar(im, ax=ax, label="Reconstructed value (legacy scaling)")
    fig.suptitle(current_case)
    plt.show()
    plt.close(fig)

## i12_dataset5

Original case: `test_reconstruct_FBP3d_tomobar_i12_dataset5`. Correct the complete projections, take their negative logarithm,
stitch with the recorded overlap and reconstruct with the matching first-half angles.

In [ ]:
def test_reconstruct_FBP3d_tomobar_i12_dataset5(i12_dataset5: tuple):
    force_clean_gpu_memory()
    projdata = i12_dataset5[0]
    angles = i12_dataset5[1]
    flats = i12_dataset5[2]
    darks = i12_dataset5[3]
    del i12_dataset5
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    del flats, darks, projdata
    force_clean_gpu_memory()
    stiched_data_180degrees = sino_360_to_180(
        data_normalised, overlap=186.66, side="left"
    )
    force_clean_gpu_memory()
    recon_data = FBP3d_tomobar(
        stiched_data_180degrees,
        np.deg2rad(angles[0:1800]),
        center=2560,
        detector_pad=0,
        filter_freq_cutoff=0.35,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (5120, 15, 5120)", recon_data.shape == (5120, 15, 5120)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_FBP3d_tomobar_i12_dataset5"
try:
    test_reconstruct_FBP3d_tomobar_i12_dataset5(load_dataset("i12_dataset5"))
except cp.cuda.memory.OutOfMemoryError as exc:
    results.append(
        {
            "test": current_case,
            "assertion": "Complete original input fits GPU",
            "passed": False,
            "details": str(exc),
        }
    )
    print(json.dumps(results[-1], indent=2))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

## i13_dataset3

Original case: `test_reconstruct_FBP3d_tomobar_i13_dataset3`. Correct the complete projections, take their negative logarithm,
stitch with the recorded overlap and reconstruct with the matching first-half angles.

In [ ]:
def test_reconstruct_FBP3d_tomobar_i13_dataset3(i13_dataset3: tuple):
    force_clean_gpu_memory()
    projdata = i13_dataset3[0]
    angles = i13_dataset3[1]
    flats = i13_dataset3[2]
    darks = i13_dataset3[3]
    del i13_dataset3
    data_normalised = dark_flat_field_correction(projdata, flats, darks)
    data_normalised = minus_log(data_normalised)
    del flats, darks, projdata
    force_clean_gpu_memory()
    stiched_data_180degrees = sino_360_to_180(
        data_normalised, overlap=438.173828, side="left"
    )
    force_clean_gpu_memory()
    recon_data = FBP3d_tomobar(
        stiched_data_180degrees,
        np.deg2rad(angles[0:3000]),
        center=2560,
        detector_pad=0,
        filter_freq_cutoff=0.35,
    )
    record_check("recon_data.flags.c_contiguous", recon_data.flags.c_contiguous)
    recon_data = cp.asnumpy(recon_data)
    record_check("recon_data.dtype == np.float32", recon_data.dtype == np.float32)
    record_check(
        "recon_data.shape == (5120, 3, 5120)", recon_data.shape == (5120, 3, 5120)
    )
    report_reconstruction(recon_data)

In [ ]:
current_case = "test_reconstruct_FBP3d_tomobar_i13_dataset3"
try:
    test_reconstruct_FBP3d_tomobar_i13_dataset3(load_dataset("i13_dataset3"))
except cp.cuda.memory.OutOfMemoryError as exc:
    results.append(
        {
            "test": current_case,
            "assertion": "Complete original input fits GPU",
            "passed": False,
            "details": str(exc),
        }
    )
    print(json.dumps(results[-1], indent=2))
force_clean_gpu_memory()
print(json.dumps([row for row in results if row["test"] == current_case], indent=2))

In [ ]:
check_results()